<a href="https://colab.research.google.com/github/Nickhilsekar/spider_ml/blob/main/Copy_of_spider_basic_taskipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# 1. Define the Custom Model Architecture
class CustomFashionModel(nn.Module):
    def __init__(self):
        super(CustomFashionModel, self).__init__()

        # Flatten layer: (None, 28, 28) -> (None, 784)
        self.flatten = nn.Flatten()

        # First Shared Hidden Layer: (None, 784) -> (None, 16)
        self.shared_hidden = nn.Linear(784, 16)
        self.relu = nn.ReLU()

        # --- LEFT BRANCH ---
        # Hidden Layer: (None, 16) -> (None, 8)
        self.left_h1 = nn.Linear(16, 8)
        # Hidden Layer: (None, 8) -> (None, 8)
        self.left_h2 = nn.Linear(8, 8)

        # --- RIGHT BRANCH ---
        # Hidden Layer: (None, 16) -> (None, 12)
        self.right_h1 = nn.Linear(16, 12)
        # Hidden Layer: (None, 12) -> (None, 8)
        self.right_h2 = nn.Linear(12, 8)

        # --- OUTPUT LAYER ---
        # Concatenated size is 8 (left) + 8 (right) = 16
        # Output Layer: (None, 16) -> (None, 10)
        self.output_layer = nn.Linear(16, 10)

    def forward(self, x):
        # Flatten and initial shared dense layer
        x = self.flatten(x)
        shared_out = self.relu(self.shared_hidden(x))

        # Left Branch with Skip Connection
        left_out1 = self.relu(self.left_h1(shared_out))
        left_out2 = self.relu(self.left_h2(left_out1))
        # Skip Connection Add: Input to branch + Output of second layer
        left_branch_final = left_out1 + left_out2

        # Right Branch
        right_out1 = self.relu(self.right_h1(shared_out))
        right_branch_final = self.relu(self.right_h2(right_out1))

        # Concatenate both branches: (None, 8) and (None, 8) -> (None, 16)
        concatenated = torch.cat((left_branch_final, right_branch_final), dim=1)

        # Final Output Layer (10 classes for Fashion-MNIST)
        out = self.output_layer(concatenated)
        return out

# 2. Data Preparation (Fashion-MNIST)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)) # Normalize pixel values
])

train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 3. Initialize Model, Loss, and Optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CustomFashionModel().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 4. Training Loop
epochs = 10
print(f"Training started on device: {device}\n" + "-"*30)

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {epoch_loss:.4f}")

# 5. Evaluation Loop
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print("-"*30 + f"\nAccuracy on Test Set: {accuracy:.2f}%")

Training started on device: cpu
------------------------------
Epoch [1/10] - Loss: 0.6813
Epoch [2/10] - Loss: 0.4616
Epoch [3/10] - Loss: 0.4217
Epoch [4/10] - Loss: 0.3998
Epoch [5/10] - Loss: 0.3836
Epoch [6/10] - Loss: 0.3731
Epoch [7/10] - Loss: 0.3630
Epoch [8/10] - Loss: 0.3549
Epoch [9/10] - Loss: 0.3493
Epoch [10/10] - Loss: 0.3420
------------------------------
Accuracy on Test Set: 85.65%
